# Polyp segmentation — reproducible Colab experiments

This notebook accompanies the `polyp-segmentation` GitHub repository. It trains the baseline U-Net/ResNet-18 model, compares BCE and BCE + soft Dice loss, evaluates the selected configuration, inspects five weak predictions, and benchmarks FP32 versus AMP inference.

**Prerequisites:**
1. A Google Colab runtime with a GPU (original results: Tesla T4, 15,360 MiB).
2. Your GitHub repository, including `src/`, `configs/splits.json`, `tests/`, and `requirements.txt`.
3. The Kvasir-SEG dataset downloaded separately from https://datasets.simula.no/kvasir-seg/ and extracted into the `images/` and `masks/` folders on Drive.
4. Corrected portable paths in the repository source scripts (see README).

Replace the repository URL in the next cell before executing. **Running the training steps may take time and will generate fresh checkpoints/metrics.** This notebook contains no model weights or dataset images.

## 1. Mount Drive and clone the repository

In [ ]:
from pathlib import Path
import subprocess
from google.colab import drive

drive.mount("/content/drive")

REPOSITORY_URL = "https://github.com/<YOUR-USERNAME>/polyp-segmentation.git"  # EDIT THIS
REPO_ROOT = Path("/content/polyp-segmentation")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPO_ROOT)], check=True)

assert (REPO_ROOT / "configs" / "splits.json").exists(), "Missing configs/splits.json"
%cd /content/polyp-segmentation

## 2. Install dependencies

In [ ]:
!python -m pip install -q -r requirements.txt
!python -c "import torch, segmentation_models_pytorch, gradio; print('CUDA available:', torch.cuda.is_available())"

## 3. Make the dataset available locally

This step **copies**, rather than moves, the extracted dataset from Drive to the Colab runtime for faster access. Modify `SOURCE_DATA` if your Drive layout differs. The dataset is not committed to Git.

In [ ]:
from pathlib import Path
import shutil

SOURCE_DATA = Path("/content/drive/MyDrive/polyp-segmentation/data/Kvasir-SEG")
TARGET_DATA = REPO_ROOT / "data" / "Kvasir-SEG"

if not TARGET_DATA.exists():
    if not SOURCE_DATA.is_dir():
        raise FileNotFoundError(
            f"Dataset not found at {SOURCE_DATA}. Download and extract Kvasir-SEG first."
        )
    TARGET_DATA.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SOURCE_DATA, TARGET_DATA)

for name in ("images", "masks"):
    folder = TARGET_DATA / name
    count = len(list(folder.glob("*.jpg")))
    print(name, count)
    assert count == 1000, f"Expected 1000 .jpg files in {folder}, got {count}"

assert (REPO_ROOT / "configs" / "splits.json").is_file()

## 4. Dataset and model smoke tests

In [ ]:
!python -m tests.test_dataset
!python -m tests.test_model

## 5. Train BCE baseline

Outputs: `checkpoints/baseline_best.pt`, `results/baseline_metrics.csv`. Training uses 800 training images and 100 validation images from the committed fixed split.

In [ ]:
!python -u -m src.train

## 6. Train BCE + soft Dice configuration

In [ ]:
!python -u -m src.train_bce_dice

## 7. Evaluate the selected model on the untouched test split

**Do not use the test metrics to tune or reselect the model.**

In [ ]:
!python -u -m src.evaluate

## 8. Visualize five lowest-Dice test images

In [ ]:

import csv
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp

from src.dataset import PolypDataset

DRIVE = REPO_ROOT
IMAGE_DIR = REPO_ROOT / "data" / "Kvasir-SEG" / "images"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read the five lowest-scoring images
with open(DRIVE / "results/test_metrics.csv") as f:
    records = list(csv.DictReader(f))

worst = sorted(records, key=lambda r: float(r["dice"]))[:5]

# Reconstruct the model and load its trained weights
model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None,
).to(device)

checkpoint = torch.load(
    DRIVE / "checkpoints/bce_dice_best.pt",
    map_location=device,
    weights_only=True,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Same normalization used during training
mean = torch.tensor(
    [0.485, 0.456, 0.406], device=device
).view(3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225], device=device
).view(3, 1, 1)

paths = [IMAGE_DIR / r["image"] for r in worst]
dataset = PolypDataset(paths)

fig, axes = plt.subplots(5, 4, figsize=(14, 17))

with torch.inference_mode():
    for i, record in enumerate(worst):
        image_tensor, mask_tensor = dataset[i]

        # Normalize and add batch dimension
        x = ((image_tensor.to(device) - mean) / std).unsqueeze(0)

        logits = model(x)
        prediction = (logits > 0).squeeze().cpu().numpy()

        image = image_tensor.permute(1, 2, 0).numpy()
        mask = mask_tensor.squeeze().numpy()

        # Original
        axes[i, 0].imshow(image)

        # Ground truth
        axes[i, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)

        # Predicted mask
        axes[i, 2].imshow(prediction, cmap="gray", vmin=0, vmax=1)

        # Overlay predicted foreground on original image
        axes[i, 3].imshow(image)
        overlay = np.ma.masked_where(~prediction, prediction)
        axes[i, 3].imshow(
            overlay, cmap="autumn", alpha=0.45, vmin=0, vmax=1
        )

        axes[i, 0].set_ylabel(
            f"Dice: {float(record['dice']):.3f}",
            fontsize=11,
        )

        for ax in axes[i]:
            ax.axis("off")

for ax, title in zip(
    axes[0],
    ["Original", "Ground Truth", "Prediction", "Overlay"],
):
    ax.set_title(title)

plt.tight_layout()

output = DRIVE / "results/worst_predictions.png"
plt.savefig(output, dpi=150, bbox_inches="tight")
plt.show()

print("Saved to:", output)


## 9. Benchmark FP32 and AMP inference (CUDA GPU required)

In [ ]:
!python -u -m src.benchmark
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 10. Archive newly generated artifacts to Google Drive

The portable source scripts save locally under the project root. This cell creates an **additional copy** in Drive for persistence. It does not touch your original experimental files stored elsewhere.

In [ ]:
from pathlib import Path
import shutil

BACKUP_ROOT = Path("/content/drive/MyDrive/polyp-segmentation/reproduced_artifacts")
for folder in ("checkpoints", "results"):
    src = REPO_ROOT / folder
    dst = BACKUP_ROOT / folder
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for file in src.iterdir():
            if file.is_file():
                shutil.copy2(file, dst / file.name)
print("Backed up artifacts to", BACKUP_ROOT)

## Reference results from the original experiment (not outputs of this notebook copy)

Original environment: **Google Colab Tesla T4 (15,360 MiB)**. Original results from five training epochs per configuration:

| Metric | BCE | BCE + soft Dice |
|---|---:|---:|
| Best validation Dice | 0.8495 | **0.8521** |
| Validation IoU at best checkpoint | 0.7637 | **0.7675** |

Selected BCE + Dice checkpoint, held-out 100-image test set: **Dice 0.8508; IoU 0.7642**.

GPU inference benchmark, batch size 1: FP32 median **7.692 ms**, AMP median **5.334 ms** (~**1.44×** faster). Real outputs are in the repository `results/` files.

**Reproducibility note:** the fixed split and random seed improve consistency but exact scores/timings may vary across GPU hardware and library builds. The dataset and model checkpoints must be obtained separately. The official Kvasir-SEG usage/citation terms apply: https://datasets.simula.no/kvasir-seg/ .